# TikTok & Local Videos → Speech + Frame OCR → AI Notes

A self-contained pipeline for short clips and other local videos in Google Colab.
Recommended runtime: **GPU T4**.

## What the notebook does

1. Looks for videos in `TikTokNotes/Videos/`, including nested folders.
2. Transcribes speech with Whisper.
3. Extracts frames with FFmpeg and recognizes on-screen text with Tesseract (`rus + eng`).
4. Caches results by file size/mtime and stage settings; unchanged videos are not reprocessed.
5. Sends the speech and frame text to Qwen to produce a Markdown note with a flexible structure and a content-value rating.

## Language parameter

`NOTE_LANGUAGE` in the first cell controls the language of the LLM prompts **and** the generated notes:

- `"ru"` — Russian prompts, notes in Russian (default);
- `"en"` — English prompts, notes in English.

All interface messages and comments in the notebook are in English either way.

## Google Drive folders

- `TikTokNotes/Videos/` — source videos.
- `TikTokNotes/Raw/` — transcribed speech.
- `TikTokNotes/FrameText/` — frame text.
- `TikTokNotes/Notes/` — Markdown notes.
- `TikTokNotes/Notes/.metadata/` — note metadata.
- `TikTokNotes/Cache/` — ASR and OCR caches.

Qwen weights are downloaded to the local Colab disk (`/content/qwen3-14b-gguf`), not to Google Drive.
After the runtime resets, the weights have to be downloaded again.

**Important:** the notebook processes video files already placed in `Videos`; there is no TikTok link downloading.

In [ ]:
# STAGE 1. SETTINGS AND ENVIRONMENT SETUP

import os, re, sys, json, time, shutil, tarfile, urllib.request, subprocess, pty, codecs
from pathlib import Path
from google.colab import drive

ROOT_DIR = Path("/content/drive/MyDrive/TikTokNotes")
INPUT_DIR = ROOT_DIR / "Videos"
RAW_DIR = ROOT_DIR / "Raw"
FRAME_TEXT_DIR = ROOT_DIR / "FrameText"
NOTES_DIR = ROOT_DIR / "Notes"
METADATA_DIR = NOTES_DIR / ".metadata"
CACHE_DIR = ROOT_DIR / "Cache"
ASR_CACHE_DIR = CACHE_DIR / "ASR"
OCR_CACHE_DIR = CACHE_DIR / "OCR"

MODEL_DIR = Path("/content/qwen3-14b-gguf")
MODEL_REPO = "Qwen/Qwen3-14B-GGUF"
MODEL_FILENAME = "Qwen3-14B-Q4_K_M.gguf"
WHISPER_MODEL = "large-v3"
ASR_LANGUAGE = "auto"    # "auto" or a language code, e.g. "ru", "en"
NOTE_LANGUAGE = "ru"     # "ru" → Russian prompts and notes; "en" → English ones
GENERATE_NOTE = True
SKIP_EXISTING = True
FRAME_INTERVAL_SECONDS = 2.0
MAX_FRAMES_PER_VIDEO = 30
FRAME_DEDUP_THRESHOLD = 6   # hamming distance of perceptual hashes for near-duplicate frames; 0 disables
USE_BATCHED_ASR = True      # batched Whisper inference (faster-whisper >= 1.x)
ASR_BATCH_SIZE = 8
AI_CONTEXT_SIZE = 32768
AI_MAX_NOTE_TOKENS = 4500
MAX_METADATA_ATTEMPTS = 4

VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v",
    ".3gp", ".mpeg", ".mpg", ".ts", ".wmv", ".flv"
}

# ------------------------------------------------------------------
# Prompt packs. The LLM prompts stay in the target language; all other
# notebook text is always in English.
# ------------------------------------------------------------------
LANGUAGE_PACK = {
    "ru": {
        "system": (
            "Ты — аналитик коротких видео и помощник по созданию базы знаний. "
            "Пиши по-русски. Сохраняй английские технические названия, названия библиотек, "
            "функций, API, игр, продуктов, мемов и оригинальные формулировки, когда их не следует переводить. "
            "Не выдумывай факты. Отличай утверждения автора от проверенных фактов. "
            "Ищи то, что имеет смысл сохранить для будущего, а не делай длинный пересказ каждого ролика."
        ),
        "content_types": {"обучение", "идея", "референс", "развлечение", "наблюдение", "другое"},
        "value_levels": {"высокая", "средняя", "низкая", "отсутствует"},
        "required_sections": [
            "Краткое содержание", "Что стоит сохранить",
            "Возможное применение", "Оценка ценности",
        ],
        "frontmatter": {"topic": "Тема", "type": "Тип", "value": "Информационная ценность"},
        "frame_marker": "[Кадр примерно {0} сек.]",
        "no_frame_text": "Текст на выбранных кадрах не обнаружен.",
        "source_labels": {
            "video": "ИМЯ ВИДЕО", "path": "ПУТЬ",
            "speech": "=== РАСПОЗНАННАЯ РЕЧЬ ===",
            "frames": "=== ТЕКСТ С КАДРОВ ===",
            "no_speech": "[Речь не распознана или отсутствует]",
            "no_frames": "[Текст на выбранных кадрах не обнаружен]",
            "truncated": "[Обрезано: превышен лимит контекста]",
        },
        "fix_hint": "Исправь ошибки предыдущего JSON: {error}",
        "metadata_prompt": """Проанализируй речь и текст с кадров короткого видео.
Верни только один корректный JSON-объект, без Markdown и пояснений.

Формат:
{
  "title": "Краткое конкретное название на русском",
  "topic": "Основная тема на русском",
  "subject": "Область или предмет на русском либо null",
  "content_type": "обучение",
  "value_level": "средняя",
  "tags": ["python", "автоматизация", "идея"]
}

Правила:
- title, topic, subject и теги — на русском, кроме имён собственных и устоявшихся технических терминов.
- content_type — одно из: "обучение", "идея", "референс", "развлечение", "наблюдение", "другое".
- value_level — одно из: "высокая", "средняя", "низкая", "отсутствует".
- Оценивай ценность для будущего: знания, саморазвитие, практические приёмы, интересные идеи и творческие референсы.
- Не считай видео бесполезным только потому, что это TikTok или в нём нет речи: визуальная идея тоже может быть референсом.
- "отсутствует" ставь, если нет полезных знаний, идеи, референса или иной причины сохранять ролик.
- tags: 3–7 коротких тегов в lowercase kebab-case. Допускаются кириллица, латиница, цифры, дефисы, а символы «+» и «#» (например c++, c#); технические термины могут оставаться английскими.
- Не добавляй неподтверждённые темы. Верни все шесть ключей.

МАТЕРИАЛ:
{{SOURCE}}""",
        "note_prompt": """Создай полезную Markdown-заметку по короткому видео.
Весь текст заметки должен быть на русском языке. Не переводи имена библиотек,
функций, API, программ, игр, брендов и технические термины, если они обычно
используются на английском. Не переводи код и оригинальные названия.

Подстраивай структуру под содержание ролика, не втискивай его в длинный универсальный шаблон.
Для технического совета сохраняй точные шаги и примеры; для идеи — суть, отличительные детали
и варианты развития; для визуального референса — композицию, стиль, цвета, объекты и детали,
которые можно использовать повторно; для развлекательного ролика — короткое описание и честную
оценку, есть ли что сохранить на будущее.

Обязательно включи четыре раздела:
# Краткое содержание
# Что стоит сохранить
# Возможное применение
# Оценка ценности

Дополнительные разделы добавляй только если они действительно подходят: например, «Пример кода»,
«Как повторить», «Визуальные детали», «Идея для собственного ролика», «Что проверить».
Не добавляй пустые разделы и не раздувай заметку ради объёма. Если полезных знаний, идеи или
референса нет, прямо укажи это; короткая заметка предпочтительнее выдуманной пользы.
Не придумывай факты. Если речь и текст на кадре противоречат друг другу, отметь это.
Не включай YAML frontmatter. Верни только Markdown, без вступления и тройных обратных кавычек.

Метаданные:
{{METADATA}}

Материал:
{{SOURCE}}""",
    },
    "en": {
        "system": (
            "You are an analyst of short videos and a knowledge-base assistant. "
            "Write in English. Keep library, function, API, game, product and meme names "
            "and original wording when they should not be translated. "
            "Do not invent facts. Distinguish the author's claims from verified facts. "
            "Look for what is worth keeping for the future rather than retelling every clip at length."
        ),
        "content_types": {"learning", "idea", "reference", "entertainment", "observation", "other"},
        "value_levels": {"high", "medium", "low", "none"},
        "required_sections": [
            "Summary", "What Is Worth Saving", "Possible Uses", "Value Assessment",
        ],
        "frontmatter": {"topic": "Topic", "type": "Type", "value": "Content value"},
        "frame_marker": "[Frame at about {0} s]",
        "no_frame_text": "No text detected on the selected frames.",
        "source_labels": {
            "video": "VIDEO NAME", "path": "PATH",
            "speech": "=== TRANSCRIBED SPEECH ===",
            "frames": "=== FRAME TEXT (OCR) ===",
            "no_speech": "[Speech not recognized or absent]",
            "no_frames": "[No text detected on the selected frames]",
            "truncated": "[Truncated: the context limit was exceeded]",
        },
        "fix_hint": "Fix the JSON errors from the previous attempt: {error}",
        "metadata_prompt": """Analyze the speech and the on-screen text of a short video.
Return a single valid JSON object only, without Markdown or explanations.

Format:
{
  "title": "Short specific title in English",
  "topic": "Main topic in English",
  "subject": "Field or subject in English, or null",
  "content_type": "learning",
  "value_level": "medium",
  "tags": ["python", "automation", "idea"]
}

Rules:
- title, topic, subject and tags are in English, except proper names and established technical terms.
- content_type is one of: "learning", "idea", "reference", "entertainment", "observation", "other".
- value_level is one of: "high", "medium", "low", "none".
- Rate the value for the future: knowledge, self-improvement, practical techniques, interesting ideas and creative references.
- Do not treat a video as useless just because it is a TikTok or has no speech: a visual idea can still be a reference.
- Use "none" when there is no useful knowledge, idea, reference or any other reason to keep the video.
- tags: 3-7 short tags in lowercase kebab-case. Latin letters, digits, hyphens are allowed, as well as "+" and "#" (e.g. c++, c#); technical terms may stay in English.
- Do not add unconfirmed topics. Return all six keys.

MATERIAL:
{{SOURCE}}""",
        "note_prompt": """Create a useful Markdown note for a short video.
The entire note must be in English. Do not translate library, function, API, program,
game or brand names and technical terms that are normally used in English.
Do not translate code and original titles.

Adapt the structure to the content of the clip instead of forcing it into a long universal template.
For a technical tip keep the exact steps and examples; for an idea — the essence, distinctive details
and ways to develop it; for a visual reference — composition, style, colors, objects and details
that can be reused; for an entertaining clip — a short description and an honest assessment of
whether anything is worth keeping.

You must include four sections:
# Summary
# What Is Worth Saving
# Possible Uses
# Value Assessment

Add extra sections only when they truly fit, for example "Code Example", "How to Reproduce",
"Visual Details", "Idea for My Own Video", "What to Check".
Do not add empty sections and do not pad the note for length. If there is no useful knowledge,
idea or reference, say so directly; a short note is better than invented value.
Do not invent facts. If the speech and the frame text contradict each other, point it out.
Do not include YAML frontmatter. Return Markdown only, with no preamble and no triple backticks.

Metadata:
{{METADATA}}

Material:
{{SOURCE}}""",
    },
}

if NOTE_LANGUAGE not in LANGUAGE_PACK:
    raise ValueError(f"NOTE_LANGUAGE must be one of {sorted(LANGUAGE_PACK)}, got {NOTE_LANGUAGE!r}")
pack_label = "Russian" if NOTE_LANGUAGE == "ru" else "English"
print(f"Note language: {NOTE_LANGUAGE} - prompts and notes in {pack_label}")

drive.mount("/content/drive")
for folder in (INPUT_DIR, RAW_DIR, FRAME_TEXT_DIR, NOTES_DIR, METADATA_DIR,
               CACHE_DIR, ASR_CACHE_DIR, OCR_CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Videos: {INPUT_DIR}")
print(f"Transcribed speech: {RAW_DIR}")
print(f"Frame text: {FRAME_TEXT_DIR}")
print(f"Notes: {NOTES_DIR}")

def run_streaming(command, env, cwd=None):
    """Runs a process through a PTY so that print and tqdm output appears in Colab immediately."""
    master_fd, slave_fd = pty.openpty()
    process = subprocess.Popen(
        command,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        env=env,
        cwd=cwd,
        close_fds=True,
    )
    os.close(slave_fd)
    decoder = codecs.getincrementaldecoder("utf-8")("replace")

    try:
        while True:
            try:
                data = os.read(master_fd, 4096)
                if not data:
                    break
                sys.stdout.write(decoder.decode(data))
                sys.stdout.flush()
            except OSError:
                # A Linux PTY usually returns EIO once the child process has exited.
                break
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        try:
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
        except Exception:
            pass
        os.close(master_fd)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Child process exited with code {return_code}: {command[0]}")
    return return_code


# FFmpeg and Tesseract with Russian and English languages
if shutil.which("ffmpeg") is None or shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq", "ffmpeg", "tesseract-ocr",
        "tesseract-ocr-rus", "tesseract-ocr-eng"
    ], check=True)

langs = subprocess.run(["tesseract", "--list-langs"], text=True, capture_output=True, check=True).stdout
if "rus" not in langs or "eng" not in langs:
    subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr-rus", "tesseract-ocr-eng"], check=True)

# Isolated Whisper environment; pyannote is not used in this pipeline.
VENV = "/content/video_whisper_env"
PYTHON = f"{VENV}/bin/python"
READY_FILE = Path(VENV) / ".video_pipeline_ready"
if not Path(PYTHON).exists() or not READY_FILE.exists():
    if Path(VENV).exists():
        shutil.rmtree(VENV)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.12", VENV], check=True)
    subprocess.run([
        "uv", "pip", "install", "--python", PYTHON,
        "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"
    ], check=True)
    get_cuda_libs = """
from pathlib import Path
import nvidia.cublas, nvidia.cudnn
print(":".join([
    str(Path(nvidia.cublas.__path__[0]) / "lib"),
    str(Path(nvidia.cudnn.__path__[0]) / "lib")
]))
"""
    result = subprocess.run(
        [PYTHON, "-c", get_cuda_libs],
        capture_output=True, text=True, check=True
    )
    cuda_libs = result.stdout.strip()
    old_path = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = cuda_libs + (":" + old_path if old_path else "")

    print("CUDA libraries linked:")
    subprocess.run([
        "uv", "pip", "install", "--python", PYTHON,
        "faster-whisper==1.2.1", "av<19", "tqdm", "huggingface_hub",
        "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"
    ], check=True)
    subprocess.run([
        PYTHON, "-c",
        "import av; print('PyAV version:', av.__version__)"
    ], check=True)
    READY_FILE.write_text("ready", encoding="utf-8")
else:
    print("The Whisper environment is already set up.")

# ============================================================
# Ready-made llama.cpp build with CUDA 12.8 for the local Qwen.
# Only the binaries are downloaded; the Qwen weights are downloaded in the last cell.
# ============================================================

LLAMA_DIR = Path("/content/llama_cpp")
LLAMA_DIR.mkdir(parents=True, exist_ok=True)


def valid_tarball(path):
    try:
        with tarfile.open(path, "r:gz") as archive:
            archive.getmembers()
        return True
    except Exception:
        return False


def download_with_progress(url, destination):
    destination = Path(destination)
    request = urllib.request.Request(url, headers={"User-Agent": "Colab-Video-Notes"})
    with urllib.request.urlopen(request, timeout=60) as response:
        total = int(response.headers.get("Content-Length", "0"))
        downloaded = 0
        with open(destination, "wb") as output:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded * 100 // total
                    print(f"\rDownloading binary: {pct}% ({downloaded // (1024 * 1024)} MiB)", end="", flush=True)
    print()


def find_llama_server():
    candidates = [p for p in LLAMA_DIR.rglob("llama-server") if p.is_file()]
    return candidates[0] if candidates else None

LLAMA_SERVER_PATH = find_llama_server()
LLAMA_READY = LLAMA_DIR / ".ready"

if GENERATE_NOTE and (LLAMA_SERVER_PATH is None or not LLAMA_READY.exists()):
    print("Looking for a ready llama.cpp build with CUDA 12.8...")
    releases_url = "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=20"
    request = urllib.request.Request(releases_url, headers={"User-Agent": "Colab-Video-Notes", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=60) as response:
        releases = json.load(response)

    selected = None
    main_asset = None
    cudart_asset = None

    for release in releases:
        assets = {asset["name"]: asset["browser_download_url"] for asset in release.get("assets", [])}
        mains = [name for name in assets if name.startswith("llama-") and "bin-ubuntu-cuda-12.8-x64.tar.gz" in name]
        for main_name in mains:
            cudart_name = "cudart-" + main_name
            if cudart_name in assets:
                selected = release
                main_asset = (main_name, assets[main_name])
                cudart_asset = (cudart_name, assets[cudart_name])
                break
        if selected:
            break

    if not selected:
        raise RuntimeError("Could not find a published llama.cpp build with CUDA 12.8. Try running this cell again later.")

    print(f"Selected llama.cpp build: {selected.get('tag_name', 'unknown')}")
    for name, url in (main_asset, cudart_asset):
        archive_path = Path("/content") / name
        if not archive_path.exists() or archive_path.stat().st_size < 1024 * 1024 or not valid_tarball(archive_path):
            if archive_path.exists():
                archive_path.unlink()
            print(f"Downloading {name}...")
            download_with_progress(url, archive_path)
        print(f"Extracting {name}...")
        with tarfile.open(archive_path, "r:gz") as archive:
            archive.extractall(LLAMA_DIR, filter="data")

    LLAMA_SERVER_PATH = find_llama_server()
    if LLAMA_SERVER_PATH is not None:
        LLAMA_READY.write_text("ready", encoding="utf-8")

if GENERATE_NOTE and LLAMA_SERVER_PATH is None:
    raise RuntimeError("llama-server executable was not found after extraction.")

LLAMA_LD_LIBRARY_PATH = ""
if GENERATE_NOTE:
    for path in LLAMA_DIR.rglob("llama-server"):
        path.chmod(path.stat().st_mode | 0o111)
    library_dirs = {str(p.parent) for p in LLAMA_DIR.rglob("*.so*") if p.is_file()}
    library_dirs.update({"/usr/local/cuda/lib64", "/usr/local/nvidia/lib64"})
    old_ld = os.environ.get("LD_LIBRARY_PATH", "")
    LLAMA_LD_LIBRARY_PATH = ":".join(sorted(library_dirs)) + (":" + old_ld if old_ld else "")

    check_env = os.environ.copy()
    check_env["LD_LIBRARY_PATH"] = LLAMA_LD_LIBRARY_PATH
    check = subprocess.run([str(LLAMA_SERVER_PATH), "--version"], env=check_env, text=True, capture_output=True)
    if check.returncode != 0:
        print(check.stdout)
        print(check.stderr)
        raise RuntimeError("Failed to run llama-server. Check CUDA access and repeat the setup.")
    print(f"llama.cpp is ready: {LLAMA_SERVER_PATH}")
else:
    print("AI is disabled: llama.cpp binaries were not installed.")



print("\nStage 1 finished.")

## Stage 2 — environment check

Whisper, GPU, FFmpeg and the Tesseract languages are verified. Large models are not loaded here.

In [ ]:
# STAGE 2. ENVIRONMENT CHECK
preflight = r"""
import sys, shutil, subprocess
print("Python:", sys.version.split()[0], flush=True)
import ctranslate2
from faster_whisper import WhisperModel
cuda_count = ctranslate2.get_cuda_device_count()
print("CUDA devices for CTranslate2:", cuda_count, flush=True)
if cuda_count < 1:
    raise RuntimeError("CUDA is unavailable for CTranslate2. Check the GPU in the Colab runtime settings.")
if not shutil.which("ffmpeg") or not shutil.which("tesseract"):
    raise RuntimeError("FFmpeg or Tesseract was not found.")
langs = subprocess.run(["tesseract", "--list-langs"], text=True, capture_output=True, check=True).stdout
if "rus" not in langs or "eng" not in langs:
    raise RuntimeError("Tesseract needs the rus and eng languages.")
print("Whisper, CUDA, FFmpeg and Tesseract: OK", flush=True)
"""
preflight_path = "/content/video_preflight.py"
Path(preflight_path).write_text(preflight, encoding="utf-8")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
run_streaming([PYTHON, "-u", preflight_path], env)

## Stage 3 — speech recognition

Whisper reads the audio straight from the video in batched mode (falls back to single-stream automatically).
The cache is tied to the video, the model and the language.

In [ ]:
# STAGE 3. WHISPER — SPEECH RECOGNITION (batched inference)
asr_script = r"""
import os, json
from pathlib import Path
from tqdm import tqdm
from faster_whisper import WhisperModel, BatchedInferencePipeline

INPUT_DIR = Path(os.environ["INPUT_DIR"])
RAW_DIR = Path(os.environ["RAW_DIR"])
CACHE_DIR = Path(os.environ["ASR_CACHE_DIR"])
MODEL_NAME = os.environ.get("MODEL_NAME", "large-v3")
LANGUAGE = os.environ.get("LANGUAGE", "auto")
LANGUAGE = None if LANGUAGE.lower() == "auto" else LANGUAGE
VIDEO_EXTENSIONS = set(os.environ["VIDEO_EXTENSIONS"].split(";"))
USE_BATCHED = os.environ.get("USE_BATCHED_ASR", "1") == "1"
BATCH_SIZE = int(os.environ.get("ASR_BATCH_SIZE", "8"))

def transcribe(audio_model, pipeline, path):
    kwargs = {"language": LANGUAGE, "vad_filter": True}
    if pipeline is not None:
        try:
            return pipeline.transcribe(str(path), batch_size=BATCH_SIZE, beam_size=5, **kwargs)
        except TypeError:
            # An older pipeline build may not accept some options; fall back to single-stream mode.
            return audio_model.transcribe(str(path), beam_size=5, **kwargs)
    return audio_model.transcribe(str(path), beam_size=5, **kwargs)

def key_for(path):
    return "__".join(path.relative_to(INPUT_DIR).with_suffix("").parts)

def fingerprint(path):
    stat = path.stat()
    return {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}

files = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
print(f"Videos found: {len(files)}", flush=True)
model = None
pipeline = None
done = skipped = failed = 0
for path in tqdm(files, desc="Whisper", unit="video", dynamic_ncols=True):
    key = key_for(path)
    output = RAW_DIR / f"{key}_raw.txt"
    manifest = CACHE_DIR / f"{key}.json"
    current = {"fingerprint": fingerprint(path), "model": MODEL_NAME, "language": LANGUAGE or "auto"}
    if output.exists() and manifest.exists():
        try:
            if json.loads(manifest.read_text(encoding="utf-8")) == current:
                print(f"Speech cache hit: {path.name}", flush=True)
                skipped += 1
                continue
        except Exception:
            pass
    print(f"\n[{done + skipped + failed + 1}/{len(files)}] {path.name}", flush=True)
    try:
        if model is None:
            print(f"Loading Whisper {MODEL_NAME}...", flush=True)
            model = WhisperModel(MODEL_NAME, device="cuda", compute_type="float16")
            if USE_BATCHED:
                try:
                    pipeline = BatchedInferencePipeline(model=model)
                    print(f"Batched mode is ON (batch size {BATCH_SIZE}).", flush=True)
                except Exception as error:
                    pipeline = None
                    print(f"Batched mode is unavailable ({type(error).__name__}); single-stream mode is used.", flush=True)
        segments, info = transcribe(model, pipeline, path)
        text = "\n".join(seg.text.strip() for seg in segments if seg.text.strip()).strip()
        output.write_text(text, encoding="utf-8")
        temp = manifest.with_suffix(".json.tmp")
        temp.write_text(json.dumps(current, ensure_ascii=False, indent=2), encoding="utf-8")
        temp.replace(manifest)
        print(f"Speech saved: {output.name}; language: {getattr(info, 'language', 'unknown')}", flush=True)
        done += 1
    except Exception as error:
        failed += 1
        print(f"ERROR {type(error).__name__}: {error}", flush=True)
print(f"\nWhisper finished. New: {done}; cached: {skipped}; errors: {failed}.", flush=True)
"""
asr_script_path = "/content/video_transcribe.py"
Path(asr_script_path).write_text(asr_script, encoding="utf-8")
asr_env = os.environ.copy()
asr_env.update({
    "INPUT_DIR": str(INPUT_DIR), "RAW_DIR": str(RAW_DIR),
    "ASR_CACHE_DIR": str(ASR_CACHE_DIR), "MODEL_NAME": WHISPER_MODEL,
    "LANGUAGE": ASR_LANGUAGE, "VIDEO_EXTENSIONS": ";".join(VIDEO_EXTENSIONS),
    "USE_BATCHED_ASR": "1" if USE_BATCHED_ASR else "0",
    "ASR_BATCH_SIZE": str(ASR_BATCH_SIZE),
    "PYTHONUNBUFFERED": "1",
})
run_streaming([PYTHON, "-u", asr_script_path], asr_env)

## Stage 4 — on-screen text recognition (OCR)

FFmpeg picks frames at a given interval and Tesseract recognizes Russian and English text.
Near-duplicate frames are detected with perceptual hashes and recognized only once (deduplication).
Only the text is stored, not the frame images. OCR runs again only when the video or the settings change.

In [ ]:
# STAGE 4. FRAME TEXT EXTRACTION (OCR) with near-duplicate frame dereplication
import sys, subprocess, tempfile

try:
    import imagehash
    from PIL import Image
except ImportError:
    print("Installing imagehash...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)
    import imagehash
    from PIL import Image

pack = LANGUAGE_PACK[NOTE_LANGUAGE]

def video_key(path):
    return "__".join(path.relative_to(INPUT_DIR).with_suffix("").parts)

def video_fingerprint(path):
    stat = path.stat()
    return {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}

SETTING_KEYS = ("fingerprint", "frame_interval_seconds", "max_frames",
                "languages", "engine", "frame_dedup_threshold")

def build_manifest(fp):
    return {
        "fingerprint": fp,
        "frame_interval_seconds": FRAME_INTERVAL_SECONDS,
        "max_frames": MAX_FRAMES_PER_VIDEO,
        "languages": "rus+eng",
        "engine": "tesseract",
        "frame_dedup_threshold": FRAME_DEDUP_THRESHOLD,
    }

def settings_match(saved, current):
    """Old caches migrate leniently: a missing key counts as equal to the current value."""
    return all(saved.get(key, current.get(key)) == current.get(key) for key in SETTING_KEYS)

videos = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
print(f"Videos found for OCR: {len(videos)}")
ocr_done = ocr_cached = ocr_failed = 0

for index, video in enumerate(videos, 1):
    key = video_key(video)
    output = FRAME_TEXT_DIR / f"{key}_frames.txt"
    manifest = OCR_CACHE_DIR / f"{key}.json"
    current = build_manifest(video_fingerprint(video))
    if output.exists() and manifest.exists():
        try:
            saved = json.loads(manifest.read_text(encoding="utf-8"))
            if settings_match(saved, current):
                print(f"[{index}/{len(videos)}] OCR cache hit: {video.name}")
                ocr_cached += 1
                continue
        except Exception:
            pass
    print(f"\n[{index}/{len(videos)}] OCR: {video.name}")
    try:
        with tempfile.TemporaryDirectory(prefix="video_frames_") as temp_dir:
            temp_path = Path(temp_dir)
            pattern = str(temp_path / "frame_%04d.jpg")
            fps = 1.0 / max(0.25, float(FRAME_INTERVAL_SECONDS))
            subprocess.run([
                "ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(video),
                "-vf", f"fps={fps:.6f},scale='min(1280,iw)':-2",
                "-frames:v", str(int(MAX_FRAMES_PER_VIDEO)), "-q:v", "3", pattern
            ], check=True, capture_output=True)
            frame_files = sorted(temp_path.glob("frame_*.jpg"))
            frame_hashes = []
            extracted, seen = [], set()
            skipped_duplicates = 0
            for frame_index, frame in enumerate(frame_files):
                # Deduplication: a perceptual hash is stable to compression and scaling,
                # so near-identical frames (static screens) are recognized only once.
                if FRAME_DEDUP_THRESHOLD > 0:
                    frame_hash = imagehash.phash(Image.open(frame))
                    if any((frame_hash - previous) <= FRAME_DEDUP_THRESHOLD for previous in frame_hashes):
                        skipped_duplicates += 1
                        continue
                    frame_hashes.append(frame_hash)
                result = subprocess.run(
                    ["tesseract", str(frame), "stdout", "-l", "rus+eng", "--psm", "6"],
                    text=True, capture_output=True, timeout=60
                )
                text = re.sub(r"[ \t]+", " ", result.stdout).strip()
                text = re.sub(r"\n{2,}", "\n", text)
                if text and text.casefold() not in seen:
                    seen.add(text.casefold())
                    extracted.append(
                        pack["frame_marker"].format(round(frame_index * FRAME_INTERVAL_SECONDS, 1)) + f"\n{text}"
                    )
        output.write_text("\n\n".join(extracted) if extracted else pack["no_frame_text"], encoding="utf-8")
        temp = manifest.with_suffix(".json.tmp")
        temp.write_text(json.dumps(current, ensure_ascii=False, indent=2), encoding="utf-8")
        temp.replace(manifest)
        print(f"Frame text saved: {output.name}; frames: {len(frame_files)}; near-duplicates skipped: {skipped_duplicates}")
        ocr_done += 1
    except Exception as error:
        ocr_failed += 1
        print(f"ERROR {type(error).__name__}: {error}")
print(f"\nOCR finished. New: {ocr_done}; cached: {ocr_cached}; errors: {ocr_failed}.")

## Stage 5 — note generation with Qwen

The model receives both the speech and the OCR text. The structure adapts to the clip; usefulness is rated separately.

Performance details: the metadata call is guided by a JSON schema (`response_format`), with retries kept as
a fallback; the KV cache of the shared prompt prefix is reused (`--cache-reuse`); when the source does not fit
the context it is trimmed — frame text first, then speech. When the source video changes, the note is considered outdated.

In [ ]:
# STAGE 5. QWEN — ANALYSIS AND NOTE GENERATION

if not GENERATE_NOTE:
    print("Note generation is disabled (GENERATE_NOTE=False).")
else:
    import socket
    import urllib.error
    from datetime import date
    from huggingface_hub import hf_hub_download

    PACK = LANGUAGE_PACK[NOTE_LANGUAGE]
    SYSTEM_PROMPT = PACK["system"]
    METADATA_PROMPT = PACK["metadata_prompt"]
    NOTE_PROMPT = PACK["note_prompt"]

    # JSON schema for the metadata call: the server constrains generation with a grammar,
    # so valid JSON is expected on the first attempt; retries stay as a fallback.
    RESPONSE_FORMAT_ENABLED = True
    METADATA_SCHEMA = {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "topic": {"type": "string"},
            "subject": {"type": ["string", "null"]},
            "content_type": {"type": "string", "enum": sorted(PACK["content_types"])},
            "value_level": {"type": "string", "enum": sorted(PACK["value_levels"])},
            "tags": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["title", "topic", "subject", "content_type", "value_level", "tags"],
    }

    def free_port():
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.bind(("127.0.0.1", 0))
            return sock.getsockname()[1]

    PORT = free_port()
    BASE_URL = f"http://127.0.0.1:{PORT}"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    def post_json(path, payload, timeout=3600):
        request = urllib.request.Request(
            BASE_URL + path,
            data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))

    def model_completion(system_prompt, user_prompt, max_tokens, temperature=0.1, json_schema=None):
        combined = system_prompt + "\n\n" + user_prompt
        try:
            prompt_tokens = len(post_json("/tokenize", {"content": combined}, timeout=300).get("tokens", []))
        except Exception:
            prompt_tokens = max(1, int(len(combined) / 1.5))
        available = CONTEXT_SIZE - prompt_tokens - 512
        if available < 256:
            raise RuntimeError(f"Input is too long for the context of {CONTEXT_SIZE} (about {prompt_tokens} tokens).")
        payload = {
            "model": "Qwen3-14B-Q4_K_M",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": temperature, "top_p": 0.9,
            "max_tokens": min(max_tokens, available), "stream": False
        }
        if json_schema is not None:
            payload["response_format"] = {"type": "json_object", "schema": json_schema}
        try:
            result = post_json("/v1/chat/completions", payload, timeout=3600)
        except urllib.error.HTTPError as error:
            # Older llama-server builds reject response_format; retry once without it.
            if json_schema is None or error.code != 400:
                raise
            global RESPONSE_FORMAT_ENABLED
            print("The server rejected response_format; falling back to plain generation.")
            RESPONSE_FORMAT_ENABLED = False
            payload.pop("response_format", None)
            result = post_json("/v1/chat/completions", payload, timeout=3600)
        choices = result.get("choices", [])
        if not choices:
            raise RuntimeError("Qwen returned a response without choices.")
        choice = choices[0]
        text = choice.get("message", {}).get("content", "").strip()
        text = re.sub("<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        if not text:
            raise RuntimeError("Qwen returned an empty response.")
        if choice.get("finish_reason") == "length":
            raise RuntimeError("The response was truncated by the token limit.")
        return text

    # "+" and "#" are allowed so that technical tags like "c++" or "c#" pass validation.
    TAG_PATTERN = r"[a-zа-яё0-9+#]+(?:-[a-zа-яё0-9+#]+)*"

    def parse_metadata(text):
        cleaned = re.sub("<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        start = cleaned.find("{")
        if start < 0:
            raise ValueError("There is no JSON object in the response.")
        data, _ = json.JSONDecoder().raw_decode(cleaned[start:])
        required = {"title", "topic", "subject", "content_type", "value_level", "tags"}
        if required - data.keys():
            raise ValueError(f"Missing keys: {sorted(required - data.keys())}")
        for key in ("title", "topic"):
            if not isinstance(data[key], str) or not data[key].strip():
                raise ValueError(f"{key} must be a non-empty string.")
            data[key] = data[key].strip()
        if data["subject"] is not None and not isinstance(data["subject"], str):
            raise ValueError("subject must be a string or null.")
        if data["content_type"] not in PACK["content_types"]:
            raise ValueError("Invalid content_type value.")
        if data["value_level"] not in PACK["value_levels"]:
            raise ValueError("Invalid value_level value.")
        tags = data["tags"]
        if not isinstance(tags, list) or not 3 <= len(tags) <= 7:
            raise ValueError("tags must contain between 3 and 7 tags.")
        normalized = []
        for tag in tags:
            if not isinstance(tag, str):
                raise ValueError("Every tag must be a string.")
            tag = tag.strip().lower()
            if not re.fullmatch(TAG_PATTERN, tag):
                raise ValueError(f"Invalid tag: {tag!r}.")
            if tag not in normalized:
                normalized.append(tag)
        if not 3 <= len(normalized) <= 7:
            raise ValueError("After removing duplicates fewer than 3 tags remained.")
        data["tags"] = normalized
        if data["subject"] is not None:
            data["subject"] = data["subject"].strip() or None
        return data

    def yaml_scalar(value):
        return "null" if value is None else json.dumps(str(value), ensure_ascii=False)

    def build_frontmatter(metadata, source_name, relative_path):
        labels = PACK["frontmatter"]
        lines = ["---", f"title: {yaml_scalar(metadata['title'])}", "tags:"]
        lines.extend(f"  - {yaml_scalar(tag)}" for tag in metadata["tags"])
        lines.extend([
            f"subject: {yaml_scalar(metadata['subject'])}",
            f"content_type: {yaml_scalar(metadata['content_type'])}",
            f"value_level: {yaml_scalar(metadata['value_level'])}",
            f"created: {yaml_scalar(date.today().isoformat())}",
            f"source_file: {yaml_scalar(source_name)}",
            f"source_path: {yaml_scalar(relative_path)}",
            "---", "",
            f"**{labels['topic']}:** {metadata['topic']}",
            f"**{labels['type']}:** {metadata['content_type']}",
            f"**{labels['value']}:** {metadata['value_level']}", ""
        ])
        return "\n".join(lines)

    def wait_for_server(server, timeout=900):
        started = time.monotonic()
        last_log = started
        while time.monotonic() - started < timeout:
            if server.poll() is not None:
                raise RuntimeError(f"llama-server exited with code {server.returncode}.")
            try:
                with urllib.request.urlopen(BASE_URL + "/health", timeout=3) as response:
                    if response.status == 200:
                        print("llama-server is ready.")
                        return
            except Exception:
                pass
            if time.monotonic() - last_log >= 15:
                print("Waiting for Qwen to load...")
                last_log = time.monotonic()
            time.sleep(2)
        raise TimeoutError("llama-server did not become ready within 15 minutes.")

    def video_key(path):
        return "__".join(path.relative_to(INPUT_DIR).with_suffix("").parts)

    def fingerprint(path):
        stat = path.stat()
        return {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}

    # Reserved for the note itself, the prompts, the metadata JSON and the KV-cache safety gap.
    SOURCE_BUDGET_TOKENS = max(1024, AI_CONTEXT_SIZE - 512 - AI_MAX_NOTE_TOKENS - 1200)
    TRUNCATED_MARK = PACK["source_labels"]["truncated"]

    def count_tokens(text):
        try:
            return len(post_json("/tokenize", {"content": text}, timeout=300).get("tokens", []))
        except Exception:
            return max(1, int(len(text) / 1.5))

    def render_source(video, speech, visual):
        labels = PACK["source_labels"]
        return (
            f"{labels['video']}: {video.name}\n"
            f"{labels['path']}: {video.relative_to(INPUT_DIR).as_posix()}\n\n"
            f"{labels['speech']}\n{speech or labels['no_speech']}\n\n"
            f"{labels['frames']}\n{visual or labels['no_frames']}"
        )

    def fit_source(video, speech, visual, budget):
        """Shrinks the source until it fits the context; frame text is cut first, then speech."""
        source = render_source(video, speech, visual)
        if count_tokens(source) <= budget:
            return source, False
        for section in ("visual", "speech"):
            for _ in range(8):
                text = visual if section == "visual" else speech
                if not text.strip():
                    break
                half = text[:max(1, int(len(text) * 0.5))]
                boundary = half.rfind("\n")
                if boundary > 0:
                    half = half[:boundary]
                half = half.strip()
                if half:
                    half = half + "\n" + TRUNCATED_MARK
                if section == "visual":
                    visual = half
                else:
                    speech = half
                if count_tokens(render_source(video, speech, visual)) <= budget:
                    return render_source(video, speech, visual), True
        return render_source(video, speech, visual), True

    videos = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
    pending = []
    for video in videos:
        key = video_key(video)
        note_path = NOTES_DIR / f"{key}.md"
        metadata_path = METADATA_DIR / f"{key}.json"
        fp = fingerprint(video)
        if SKIP_EXISTING and note_path.exists() and metadata_path.exists():
            try:
                cached = json.loads(metadata_path.read_text(encoding="utf-8"))
                if cached.get("_source_fingerprint") == fp:
                    print(f"Note is up to date, skipping: {video.name}")
                    continue
            except Exception:
                pass
        raw_path = RAW_DIR / f"{key}_raw.txt"
        frame_path = FRAME_TEXT_DIR / f"{key}_frames.txt"
        speech = raw_path.read_text(encoding="utf-8", errors="replace").strip() if raw_path.exists() else ""
        visual = frame_path.read_text(encoding="utf-8", errors="replace").strip() if frame_path.exists() else ""
        if not speech and not visual:
            print(f"Skipping: no transcript and no OCR text: {video.name}")
            continue
        pending.append((video, key, note_path, metadata_path, speech, visual, fp))

    print(f"Videos to process for note generation: {len(pending)}")
    if pending:
        model_path = MODEL_DIR / MODEL_FILENAME
        if not model_path.exists() or model_path.stat().st_size < 8_000_000_000:
            print(f"Downloading Qwen to the local Colab disk: {MODEL_REPO}/{MODEL_FILENAME}")
            try:
                model_path = Path(hf_hub_download(
                    repo_id=MODEL_REPO, filename=MODEL_FILENAME, local_dir=str(MODEL_DIR)
                ))
            except Exception:
                if model_path.exists() and model_path.stat().st_size < 8_000_000_000:
                    model_path.unlink()
                raise
        else:
            print(f"Qwen weights are already present: {model_path}")

        server_env = os.environ.copy()
        server_env["LD_LIBRARY_PATH"] = LLAMA_LD_LIBRARY_PATH
        server_env["CUDA_VISIBLE_DEVICES"] = "0"
        server_log = open("/content/llama-server-video.log", "w", encoding="utf-8")
        server = None
        try:
            base_command = [
                str(LLAMA_SERVER_PATH), "--model", str(model_path),
                "--host", "127.0.0.1", "--port", str(PORT),
                "--alias", "Qwen3-14B-Q4_K_M",
                "--ctx-size", str(AI_CONTEXT_SIZE), "--n-gpu-layers", "99",
                "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
                # Reuse the KV cache of the shared prompt prefix across requests.
                "--cache-reuse", "256",
                "--batch-size", "256", "--ubatch-size", "128",
                "--parallel", "1", "--reasoning", "off"
            ]
            selected_context = None
            for candidate in dict.fromkeys([AI_CONTEXT_SIZE, 24576, 16384]):
                command = list(base_command)
                command[command.index("--ctx-size") + 1] = str(candidate)
                print(f"Starting Qwen with a context of {candidate}...")
                server = subprocess.Popen(command, env=server_env, stdout=server_log, stderr=subprocess.STDOUT)
                try:
                    wait_for_server(server)
                    selected_context = candidate
                    break
                except Exception as error:
                    if server.poll() is None:
                        server.terminate()
                        try:
                            server.wait(timeout=15)
                        except subprocess.TimeoutExpired:
                            server.kill()
                            server.wait()
                    server = None
                    print(f"Context of {candidate} failed to start: {error}")
            if server is None or selected_context is None:
                raise RuntimeError("Failed to start Qwen. See /content/llama-server-video.log")
            CONTEXT_SIZE = selected_context

            completed = failed = 0
            for index, (video, key, note_path, metadata_path, speech, visual, fp) in enumerate(pending, 1):
                print(f"\n[{index}/{len(pending)}] {video.name}")
                source, source_truncated = fit_source(video, speech, visual, SOURCE_BUDGET_TOKENS)
                if source_truncated:
                    print("Source was trimmed to fit the context (frame text first, then speech).")
                metadata = None
                if metadata_path.exists():
                    try:
                        cached = json.loads(metadata_path.read_text(encoding="utf-8"))
                        if cached.get("_source_fingerprint") == fp:
                            metadata = parse_metadata(json.dumps(cached, ensure_ascii=False))
                            print("Using cached metadata.")
                    except Exception:
                        metadata = None

                if metadata is None:
                    previous_error = ""
                    for attempt in range(1, MAX_METADATA_ATTEMPTS + 1):
                        print(f"Metadata: attempt {attempt}/{MAX_METADATA_ATTEMPTS}")
                        prompt = METADATA_PROMPT.replace("{{SOURCE}}", source)
                        if previous_error:
                            prompt += "\n\n" + PACK["fix_hint"].format(error=previous_error)
                        try:
                            schema = METADATA_SCHEMA if RESPONSE_FORMAT_ENABLED else None
                            metadata = parse_metadata(model_completion(SYSTEM_PROMPT, prompt, 900, 0.0, json_schema=schema))
                            saved = dict(metadata)
                            saved["_source_fingerprint"] = fp
                            temp = metadata_path.with_suffix(".json.tmp")
                            temp.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
                            temp.replace(metadata_path)
                            break
                        except Exception as error:
                            previous_error = str(error)
                            print(f"Metadata failed validation: {error}")
                    if metadata is None:
                        failed += 1
                        print("Could not obtain valid metadata.")
                        continue

                note_prompt = (NOTE_PROMPT
                               .replace("{{METADATA}}", json.dumps(metadata, ensure_ascii=False, indent=2))
                               .replace("{{SOURCE}}", source))
                try:
                    note_body = model_completion(SYSTEM_PROMPT, note_prompt, AI_MAX_NOTE_TOKENS, 0.2)
                    for heading in PACK["required_sections"]:
                        if not re.search(r"(?m)^#\s+" + re.escape(heading) + r"\s*$", note_body):
                            note_body += f"\n\n# {heading}\n\nThe section was not generated by the model."
                    note_text = build_frontmatter(
                        metadata, video.name, video.relative_to(INPUT_DIR).as_posix()
                    ) + note_body.strip() + "\n"
                    temp = note_path.with_suffix(".md.tmp")
                    temp.write_text(note_text, encoding="utf-8")
                    temp.replace(note_path)
                    completed += 1
                    print(f"Note saved: {note_path}")
                except Exception as error:
                    failed += 1
                    print(f"ERROR {type(error).__name__}: {error}")
            print(f"\nAI finished. Completed: {completed}; errors: {failed}.")
        finally:
            if server is not None and server.poll() is None:
                print("Stopping llama-server...")
                server.terminate()
                try:
                    server.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    server.kill()
                    server.wait()
            server_log.close()
    else:
        print("No new or changed videos for note generation.")